Object Localization(simpler case):

- Finding WHAT and WHERE a single object exists in an image.

Object Detection:

- Finding WHAT and WHERE multiple objects are in an image.

Intersection over Union: (prediction boundry box and real boundry box)

IoU = Area of Intersection / Area of Union

IoU > 0.5 "decent"

IoU > 0.7 "pretty good"

IoU > 0.9 "almost perfect"

Box1 = [x1,y1, x2,y2] for example


In [ ]:
import torch

def intersection_over_union(boxes_preds, boxes_labels, box_format="corners"):
    # (N, 4) where N is the number of bboxes
        
    if box_format == "corners":
        box1_x1 = boxes_preds[..., 0:1] # slicing preserves the shape of a tensor (N, 1) instead of (N)
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]
    
    else:
        raise NotImplementedError("Only corners format is supported.")    
    
    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.max(box1_x2, box2_x2)
    y2 = torch.max(box1_y2, box2_y2)
    
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    
    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))
    
    return intersection / (box1_area + box2_area - intersection + 1e-6)

Non Max Suppression(Cleaning up Bounding Boxes): At the end I will have only one bbox.

(Perhaps start with discarding all bboxes < Probability Threshold)

While BoundingBoxes:

    - Take out the largest probability box

    - Remove all other boxes with IoU > threshold

(And we do this for each class)




In [1]:
def non_max_supression(
    bboxes,
    iou_threshold,
    prob_threshold,
    box_format="corners"
):
    # predictions = [[1, 0.9, x1, y1, x2, y2], [], []] # [class, prob, left top, right down]
    assert type(bboxes) == list
    
    bboxes = [box for box in bboxes if box[1] > prob_threshold]
    
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)
    
    bboxes_after_nms = []
    
    while bboxes:
        
        chosen_box = bboxes.pop(0)
        
        bboxes = [
            box
            for box in bboxes
            if box[0] != chosen_box[0]
            or intersection_over_union(torch.tensor(box[2:]), torch.tensor(chosen_box[2:])) < iou_threshold
        ]
        
        bboxes_after_nms.append(chosen_box)
        
    return bboxes_after_nms

Mean Average Precision(The most common metric used in Deep Learning to evaluate object detection models):

Precision and Recall

Precision = True Positive / True Positive + False Positive

Recall = True Positive / Total number of target bounding boxes

![alt text](image1-1.png)

![](image2.png)


Green Ones are the target bboxes, and red ones are the predicted bboxes

So there are 4 TARGET BBOXES, and 7 PREDICTED BBOXES

![](image3.png)


![alt text](image4.png)

So the Precision is 3 / 3 + 4 = 3/7

And the Recall is 3/4

In [1]:
from collections import Counter


def mean_average_precision(
    pred_boxes,
    true_boxes,
    iou_threshold=0.5,
    box_format="corners",
    num_classes=20
):
    # pred_boxes (list): [[train_idx, class_pred, prob_score, x1, y1, x2, y2], ...]
    average_precisions = []
    epsilon = 1e-6
    
    for c in range(num_classes):
        
        detections = []
        ground_truths = []
        
        for detection in pred_boxes:
            if detection[1] == c:
                detections.append(detection)
                
        for true_box in true_boxes:
            if true_box[1] == c:
                ground_truths.append(true_box)
                
        amount_bboxes = Counter([gt[0] for gt in ground_truths])
        
        for k,v in amount_bboxes.items():
            amount_bboxes[k] = torch.zeros(v)
            
        # amount_boxes = {0:torch.tensor([0,0,0]), 1:torch.tensor([0,0,0,0,0])}
        
        detections.sort(key=lambda x: x[2], reverse=True)
        TP = torch.zeros(len(detections))
        FP = torch.zeros(len(detections))
        total_true_bboxes = len(ground_truths)
        
        for detection_idx, detection in enumerate(detections):
            
            ground_truth_img = [bbox for bbox in ground_truths if bbox[0] == detection[0]] # all ground truths within the same img
        
            num_gts = len(ground_truth_img)
            best_iou = 0
            
            for idx, gt in enumerate(ground_truth_img):
                iou = intersection_over_union(
                    torch.tensor(detection[3:]),
                    torch.tensor(gt[3:])
                )
                
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx
            
            if best_iou > iou_threshold:
                if amount_bboxes[detection[0]][best_gt_idx] == 0:
                    TP[detection_idx] = 1
                    amount_bboxes[detection[0]][best_gt_idx] = 1
                else:
                    FP[detection_idx] = 1
            
            else:
                
                FP[detection_idx] = 1
                
        TP_cumsum = torch.cumsum(TP, dim=0)      
        FP_cumsum = torch.cumsum(FP, dim=0)
        precisions = TP_cumsum / (TP_cumsum + FP_cumsum + epsilon)  
        recalls = TP_cumsum / (total_true_bboxes + epsilon)
        
        precisions = torch.cat(torch.tensor([1], precisions))
        recalls = torch.cat(torch.tensor([0]), recalls)
        
        average_precisions.append(torch.trapz(precisions, recalls))
        
    return sum(average_precisions) / len(average_precisions)
        
                    

IMPLEMENTATION OF MY CNN OBJECT LOCALIZATION MODEL
